# Olist Brazilian E-Commerce Analytics
## Project 1 · P1-01 — Business Scope & Questions

**Purpose:** translate business interests into measurable, decision-oriented questions that can be answered from the Olist public dataset.

**Working environment:** Jupyter Notebook in VS Code · Python · Pandas · DuckDB

**Task boundary:** this notebook covers business scoping, metric specification, schema feasibility, stakeholder ownership, and table/grain mapping. It is **not** the data-ingestion, cleaning, full profiling, KPI computation, or visualization notebook.

### Expected P1-01 deliverables

1. A concise set of business questions with explicit metrics and populations.
2. A named decision-maker and plausible operational/commercial decision for each question.
3. A source-column and table mapping for each question.
4. Documented feasibility constraints, join-grain risks, and assumptions.
5. A short operational response to the late-delivery scenario in the task guide.

> **Professional standard:** do not call a question feasible merely because a similarly named column exists. Confirm its business meaning, table grain, join path, population, and limitations.

## 0. Working conventions and scope guardrails

### Conventions

- Treat source CSV files as immutable inputs. Do not overwrite raw source data.
- Use descriptive `snake_case` names for variables and fields created later.
- Use `order_id`, `customer_id`, and `customer_unique_id` according to their distinct meanings; never treat them as interchangeable.
- State the grain of every metric before choosing an aggregation.
- Make metric definitions explicit: numerator, denominator, eligible population, time attribution, unit, and segmentation.
- Record assumptions and unresolved questions rather than silently making them.
- Separate observed facts from interpretations and recommendations.

### In scope for P1-01

- Review relevant schema metadata and column semantics.
- Define business questions and metric contracts.
- Identify decision owners and possible actions.
- Verify column availability and join paths.
- Document table grain, cardinality risks, and limitations.

### Out of scope for P1-01

- Full raw-data ingestion and transformation pipeline.
- Data cleaning or imputation.
- Comprehensive exploratory data analysis or automated profiling reports.
- Final metric calculations, statistical modelling, dashboards, or causal claims.

**Instructor note:** resist the temptation to build the entire analysis now. Your job here is to specify what should be measured and prove that the data can support it.

## 1. Environment and imports

Use the existing project environment. Do not upgrade packages in this notebook merely to follow a trend; keep the environment reproducible.

- **Pathlib:** construct and inspect filesystem paths without changing the working directory.
- **Pandas:** represent small metadata/specification tables and selected results.
- **DuckDB:** inspect CSV schemas and perform relational checks with SQL without requiring a separate database server.

**Tooling decision:** Pandas + DuckDB are sufficient for this dataset and task. Polars is a credible alternative for larger or more transformation-heavy workloads, but introducing it here adds another API without a clear benefit. Pydantic is best suited to validating structured objects/configuration; for dataframe/table contracts, consider Pandera or explicit SQL assertions in a later validation task.

In [ ]:
# Standard project imports — setup only; no analytical logic is implemented here.
from pathlib import Path  # Portable path construction and filesystem inspection
import pandas as pd  # DataFrame representation for metric specifications and selected results
import duckdb  # SQL engine for inspecting and querying CSV/Parquet data

## 2. Dataset inventory and schema review

This is a **scope-level schema review**, not full data loading or profiling.

### 2.1 Relevant source tables

| Source file / logical table | Relevant role | Expected analytical grain |
|---|---|---|
| `olist_orders_dataset` / `orders` | Order status and lifecycle timestamps | One row per order |
| `olist_customers_dataset` / `customers` | Customer identity and delivery geography | One row per order-specific `customer_id` |
| `olist_order_payments_dataset` / `payments` | Payment method, installments, payment value | One row per payment record |
| `olist_order_items_dataset` / `order_items` | Products and sellers purchased in an order | One row per order-item line |
| `olist_products_dataset` / `products` | Product attributes and Portuguese category | One row per product |
| `product_category_name_translation` | Portuguese-to-English category mapping | One row per category mapping |
| `olist_order_reviews_dataset` / `reviews` | Review score and review metadata | One row per review record; verify order-level cardinality |

The canonical dataset schema and relationships are documented by the dataset publisher and public schema references. Verify against the actual files in your project rather than assuming that every downloaded copy has identical headers or row counts.

### 2.2 Order lifecycle semantics

| Column | Meaning | Use in this task? |
|---|---|---|
| `order_purchase_timestamp` | Time the order was placed | Yes — sales/payment trend time attribution |
| `order_approved_at` | Time the order was approved | Review as a lifecycle milestone; not the delivery deadline |
| `order_delivered_carrier_date` | Time the order was handed to the logistics partner | Yes — understand the handover milestone; not final delivery |
| `order_delivered_customer_date` | Actual customer delivery timestamp | Yes — actual delivery outcome |
| `order_estimated_delivery_date` | Estimated customer delivery deadline | Yes — comparison baseline |
| `order_status` | Current order status | Yes — define the eligible delivered-order population |

**Critical distinction:** carrier handover is not delivery to the customer. The delivery-promise question must compare actual customer delivery with the estimated customer delivery deadline.

### 2.3 Schema review tasks

In [ ]:
# TODO 2.1: Identify the raw-data directory using pathlib.
# TODO: Check that the specific Olist source files needed for P1-01 are present.
# TODO: Do not move, rename, overwrite, or clean the source files.
# TODO: Store the resolved input paths in clearly named variables.

#TODO 2.1: Identify the raw-data directory using pathlib :
from pathlib import path 

# Identify the raw-data directory (current working directory of the notebook)
raw_data_dir = path.cwd()

# Verify the path:
print(f"the raw data directory is: " {raw_data_dir})



In [ ]:
# TODO 2.2: Use DuckDB metadata/schema inspection to inspect the relevant CSV headers.
# TODO: Confirm the exact column names required by the five business questions.
# TODO: Compare the actual headers with the expected names documented above.
# TODO: Record missing, renamed, or unexpected columns; do not silently substitute fields.

In [ ]:
# TODO 2.3: Inspect the data types / timestamp representation of the order lifecycle fields.
# TODO: Determine whether the relevant fields contain time-of-day values or date-only values.
# TODO: Check whether timezone information is present; do not assume a timezone.
# TODO: Document the chosen business rule for comparing actual delivery with the estimate.
# TODO: Do not perform full timestamp cleaning or imputation in P1-01.

## 3. Metric specification standard

Before writing a question, define its metric contract. Every metric specification in this notebook should state:

| Attribute | What must be documented |
|---|---|
| Business question | The stakeholder-facing question the analysis will answer |
| Decision owner | The person/team able to act on the finding |
| Metric name | A precise, unambiguous label |
| Numerator | The count/value in the numerator, where applicable |
| Denominator | The eligible population in the denominator, where applicable |
| Grain | What one input/metric row represents |
| Population and filters | Which records are included and excluded, and why |
| Time attribution | Which timestamp assigns an observation to a period |
| Segmentation | State, category, delivery cohort, or other relevant dimension |
| Source fields and tables | Exact columns and source files required |
| Join keys and cardinality | Join path and expected one-to-one / one-to-many relationship |
| Limitations | Missing values, unobserved factors, and claims the data cannot support |

Do not treat a metric as production-ready until these fields are sufficiently defined.

## 4. Business question A — Delivery promise reliability

### Proposed stakeholder question

> Which customer states have the highest late-delivery rates, and what are the average and median delays among late delivered orders?

### Metric contract to complete

- **Primary metric:** late-delivery rate.
- **Numerator:** eligible delivered orders whose actual customer delivery occurs after the estimated delivery deadline.
- **Denominator:** delivered orders with both actual and estimated delivery timestamps available.
- **Supporting metrics:** eligible order count; average and median delay among late orders.
- **Grain:** one order.
- **Segmentation:** `customer_state`.
- **Core source:** `orders`; join `customers` to obtain destination state.
- **Join key:** `orders.customer_id = customers.customer_id`.
- **Eligibility:** define and document the delivered-order and non-null timestamp conditions.
- **Comparison rule:** explicitly decide whether the service promise is a calendar-date deadline or an exact timestamp deadline. Use one consistent rule.

### Interpretation guardrail

Report the eligible order count and missing-timestamp coverage alongside the rate. A percentage without its denominator and coverage can be misleading.

In [ ]:
# TODO 4.1: Create the metric specification for delivery promise reliability.
# TODO: Record the business question, decision owner, numerator, denominator, grain,
# TODO: eligible population, segmentation, timestamp rule, required columns, source tables,
# TODO: join key, and known limitations. Do not calculate the final KPI in this task.

## 5. Business question B — Sales / payment performance

### Proposed stakeholder question

> How does gross recorded customer payment value change over purchase time, and how does the number of distinct orders evolve alongside it?

### Metric contract to complete

- **Primary measure:** sum of recorded `payment_value`.
- **Companion measure:** distinct order count.
- **Optional derived measure:** recorded payment value per distinct order, with its definition clearly labelled.
- **Time attribution:** `orders.order_purchase_timestamp`.
- **Tables:** `orders` and `order_payments`.
- **Join key:** `order_id`.
- **Payment grain:** one payment record, not necessarily one order.

### Financial terminology guardrail

Use terms such as **gross recorded payment value** or **customer payment volume**. The presence of `payment_value` alone does not establish accounting revenue, net revenue, contribution margin, or profit. The dataset does not provide all costs required for profitability analysis.

### Join guardrail

Do not count payment rows as orders. Do not join raw payment rows and raw order-item rows together and then sum both facts without controlling their grains; a many-to-many multiplication can inflate totals. Aggregate each child fact to a compatible grain before combining them.

In [ ]:
# TODO 5.1: Create the metric specification for payment performance.
# TODO: Record payment_value's source table and transaction grain.
# TODO: Specify purchase timestamp as the reporting-time field.
# TODO: Define how distinct orders will be counted when an order has multiple payment rows.
# TODO: State the intended order-status / payment-record inclusion policy.
# TODO: Document why this metric is not automatically net revenue or profit.

## 6. Business question C — Product category mix

### Proposed stakeholder question

> Which English product categories account for the largest share of purchased order-item lines?

### Metric contract to complete

- **Primary metric:** count of order-item rows / lines.
- **Optional mix metric:** category order-item count divided by all eligible order-item lines.
- **Grain:** one order-item line, identified within an order by `(order_id, order_item_id)`.
- **Tables:** `order_items` → `products` → `product_category_name_translation`.
- **Join keys:** `order_items.product_id = products.product_id`; `products.product_category_name = translation.product_category_name`.
- **Category label:** `product_category_name_english` comes from the translation table.

### Interpretation guardrail

An order can contain multiple item lines. Item-line count is not the same as order count, customer count, or necessarily physical unit quantity. Define how null or untranslated categories will be represented; do not silently drop them.

In [ ]:
# TODO 6.1: Create the metric specification for category mix.
# TODO: Identify the item-line grain and the exact category translation join path.
# TODO: Define the category-count numerator and overall item-line denominator.
# TODO: Decide how missing / untranslated category labels will be represented.
# TODO: Do not describe item-line counts as revenue or number of orders.

## 7. Business question D — Delivery experience and review scores

### Proposed stakeholder question

> How does the distribution of review scores differ between orders delivered on time and orders delivered late?

### Metric contract to complete

- **Outcome field:** `review_score` (scale 1–5).
- **Exposure / grouping field:** on-time versus late delivery, derived from the agreed delivery rule.
- **Possible summaries:** review-score distribution, average score, and share of low ratings (for example, scores 1–2).
- **Tables:** `orders` and `order_reviews`.
- **Join key:** `order_id`.
- **Intended conceptual grain:** one order-level delivery outcome associated with a review observation.

### Cardinality and interpretation guardrails

- Verify whether `order_id` is unique in the reviews table before joining. If multiple review records can exist for an order, document an aggregation or selection policy.
- Define how orders without a review are handled; do not silently treat them as zero-star reviews.
- This is observational analysis. A difference in review scores is an association, not proof that delivery delay caused the score difference.

In [ ]:
# TODO 7.1: Create the metric specification for delivery experience / review scores.
# TODO: Record the review outcome, delivery grouping, eligible population, and summary measures.
# TODO: Specify the order_id join and the review-to-order cardinality check required.
# TODO: Define the treatment of missing reviews and multiple reviews per order.
# TODO: State explicitly that the analysis cannot establish causation by itself.

## 8. Business question E — Repeat customer behaviour

### Proposed stakeholder question

> What proportion of identifiable customers placed at least two distinct orders during the dataset's observation period?

### Metric contract to complete

- **Customer identifier:** `customer_unique_id`.
- **Order identifier:** `order_id`.
- **Repeat-buyer rule:** at least two distinct orders associated with one `customer_unique_id` during the observed period.
- **Denominator:** identifiable unique customers with at least one observed order.
- **Tables:** `orders` and `customers`.
- **Join key:** `orders.customer_id = customers.customer_id`.
- **Aggregation grain:** one `customer_unique_id` after associating orders with customers.

### Interpretation guardrail

`customer_id` is order-specific in this dataset; `customer_unique_id` is the identifier intended to connect purchases by the same underlying customer. The result measures repeat purchasing **within the available observation window**, not lifetime retention. Customers entering near the end of the period have less observed time to make another purchase.

In [ ]:
# TODO 8.1: Create the metric specification for repeat customers.
# TODO: Specify customer_unique_id as the customer-level identifier and order_id as the order identifier.
# TODO: Define the >= 2 distinct-order repeat-buyer rule and the denominator.
# TODO: Identify the orders-to-customers join and resulting customer-level grain.
# TODO: Document the observation-window / right-censoring limitation.

## 9. Stakeholder and decision mapping

A business question is not complete until the person/team able to act on the answer is identified. Assign ownership based on the operational process observed in the data, not on an assumed field that does not exist.

| Question area | Likely decision owner | Potential decision informed by the analysis |
|---|---|---|
| Delivery reliability | Regional logistics / fulfilment operations manager | Investigate regional handover delays, delivery capacity, routing, or promise accuracy |
| Payment performance | Commercial operations / finance operations | Review order and payment-volume trends; investigate unusual movements |
| Category mix | Merchandising / category manager | Review category assortment, availability, and category-level priorities |
| Reviews vs delivery | Customer experience / service-quality lead, with logistics operations | Prioritise service recovery and investigate operational drivers associated with low scores |
| Repeat customers | CRM / customer-retention lead | Identify the scale of observed repeat purchasing and where retention work may be relevant |

**Ownership caveat:** the dataset's orders table does not include a carrier identifier. Do not claim that this analysis can identify the worst-performing carrier. Customer state identifies the destination geography, not the carrier responsible for each delivery.

In [ ]:
# TODO 9.1: Build a stakeholder mapping for all five business questions.
# TODO: For each question, record the accountable decision owner and why that team owns the process.
# TODO: Record at least one realistic decision or operational lever the finding could inform.
# TODO: Flag recommendations that the available fields cannot support (for example, carrier ranking).

## 10. Data feasibility, lineage, and join-grain audit

### 10.1 Business-question traceability

For every question, complete the following path:

`Question → Metric → Required columns → Source table(s) → Join key(s) → Grain → Population → Limitation`

### 10.2 Relationship map

| Relationship | Expected relationship to validate | Main risk |
|---|---|---|
| `orders` → `customers` on `customer_id` | One order to one customer record | Missing or duplicate customer key |
| `orders` → `order_payments` on `order_id` | One order to one or more payment records | Payment rows mistaken for order rows |
| `orders` → `order_items` on `order_id` | One order to one or more item lines | Item rows mistaken for order rows |
| `orders` → `order_reviews` on `order_id` | Validate actual review cardinality | Review rows multiplying order-level records |
| `order_items` → `products` on `product_id` | Many item lines to one product record | Missing product match or duplicated product key |
| `products` → category translation on `product_category_name` | Many products to one category mapping | Untranslated or missing category labels |

### 10.3 Join design rules

- Establish the grain of each source before joining.
- Validate uniqueness on the supposed one-side of a relationship.
- Compare row counts and distinct business-key counts before and after joins.
- Aggregate one-to-many child tables to the intended reporting grain before combining multiple fact tables.
- Do not join `order_items` and `order_payments` at raw grain and then sum both measures without a deliberate grain-safe design.
- Preserve unmatched records during feasibility checks so missing mappings become visible.

In [ ]:
# TODO 10.1: Build a traceability matrix with one row per business question.
# TODO: Include metric, numerator / denominator, grain, population, time field, dimensions,
# TODO: source tables, source columns, join keys, decision owner, and limitation.
# TODO: Use a consistent structure so that another analyst can review the specification.

In [ ]:
# TODO 10.2: Define the relationship / cardinality checks required for each join.
# TODO: Identify the key expected to be unique on each one-side.
# TODO: Specify checks for duplicate keys, unmatched keys, and unexpected row multiplication.
# TODO: Pay particular attention to payments, order items, and reviews.
# TODO: Do not combine raw one-to-many facts and aggregate their measures without controlling grain.

In [ ]:
# TODO 10.3: Produce a feasibility decision for each question: feasible, feasible with caveats, or not supported.
# TODO: Justify each decision using exact columns, source tables, join path, and metric grain.
# TODO: Identify any requested metric that requires data not present in this dataset.
# TODO: Explicitly rule out unsupported claims such as profit, marketing ROI, or carrier ranking.

## 11. Task guide — Operational response to late deliveries

### Scenario

The analysis finds that 20% of eligible delivered orders in a particular customer state arrived after the estimated delivery deadline.

### Decision-oriented response structure

Your recommendation should identify:

1. **Accountable owner:** normally the regional logistics / fulfilment operations manager, working with the relevant seller-operations and last-mile teams.
2. **Evidence to validate:** the eligible order count, late-order count, timestamp completeness, observation period, severity of delay, and comparison with other states and the overall baseline.
3. **Root-cause investigation:** separate the time before carrier handover from the time between carrier handover and customer delivery. Use the milestones available in the orders table.
4. **Potential operational levers:** seller dispatch and handover SLAs, pickup cut-off times and frequency, regional capacity allocation, route or hub constraints, and customer-facing delivery-promise accuracy.
5. **Measurement of intervention:** monitor the late-delivery rate and delay distribution again after a change, using a comparable population and time window.

### Important limitation

The dataset does not identify the individual carrier in the orders table. A state-level finding can justify a regional investigation, but it cannot by itself attribute the problem to a particular carrier. Do not recommend a carrier penalty or contract change without carrier-level evidence.

### Your response

In [ ]:
# TODO 11.1: Write a concise stakeholder-ready response to the scenario.
# TODO: Name the decision owner and explain why that role is accountable.
# TODO: State what evidence must be checked before action is taken.
# TODO: Recommend realistic operational changes linked to the observed lifecycle milestones.
# TODO: Explain what the dataset cannot attribute (especially individual carrier performance).
# TODO: Describe how the intervention's impact would be measured.

## 12. P1-01 quality gate and sign-off

Do not mark this task complete until each condition is satisfied.

- [ ] Each question is framed as a stakeholder decision, not just an interesting calculation.
- [ ] Each metric has a defined numerator, denominator where applicable, grain, population, and unit.
- [ ] The timestamp used for time attribution is explicitly identified.
- [ ] Every required column has been verified against the actual source schema.
- [ ] Every question has a named decision owner and a plausible decision lever.
- [ ] Every source table and join key is documented.
- [ ] One-to-many relationships and possible join fan-out have been considered.
- [ ] Missing timestamps, missing reviews, and untranslated categories have explicit handling policies.
- [ ] The analysis does not claim profit, net revenue, marketing ROI, or carrier-level performance without supporting data.
- [ ] Repeat buying uses `customer_unique_id` and is described as behaviour within the observation window.
- [ ] Review-score findings are described as associations, not causal effects.
- [ ] The operational guide response includes validation, root-cause investigation, action, and post-intervention measurement.

### Final P1-01 deliverable

Create a compact, reviewable business-question register and a short scope note. The register should be clear enough that another analyst could implement the next task without needing to reinterpret what each question means.

### Decision log

Record unresolved assumptions, decisions made, and any schema differences discovered in your local copy of the dataset.

In [ ]:
# TODO 12.1: Complete the P1-01 quality gate above.
# TODO: Summarise the final approved business questions and metric definitions.
# TODO: Record unresolved assumptions and decisions in a short decision log.
# TODO: Confirm that the notebook has remained within the P1-01 scope.